In [ ]:
# Phase 4 — Multi-Agent Orchestration

This notebook builds OurBrain's multi-agent system on top of the MCP server from Phase 2.

**Architecture:**
- `agents/mcp_host.py` — custom MCP host that connects to ourbrain_server.py
- `agents/agent_runner.py` — agentic loop with tool scoping
- `agents/agents.py` — three specialized sub-agents (Nutrition, Planner, Suggestion)
- `agents/orchestrator.py` — routes questions and synthesizes responses

**What this demonstrates:** custom MCP host implementation, the agentic loop pattern, 
tool scoping per agent, and orchestrator-worker multi-agent design.

In [3]:
import sys

# Add the project root (one level up from the notebook) to Python's import path
# This lets Python find the `agents` package
if ".." not in sys.path:
    sys.path.insert(0, "..")

# If anything from the old in-notebook version is cached, clear it
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.mcp_host import OurBrainMCPHost
print("Imported OurBrainMCPHost from agents.mcp_host ✓")

Imported OurBrainMCPHost from agents.mcp_host ✓


In [4]:
host = OurBrainMCPHost(server_path="../mcp_server/ourbrain_server.py")

tools = await host.list_tools()
for t in tools:
    print(f"- {t.name}: {t.description}")

- get_meals: Get all meals stored in OurKitchen, including name, cuisine, proteins, and ingredients.
- get_meal_history: Get the full weekly meal history, with each planned meal enriched with its recipe details.
- get_preferences: Get household dietary preferences and restrictions.
- get_groceries: Get the current grocery list.


In [ ]:
## Piece 2 — Agent Runner

Built `agents/agent_runner.py` containing `run_agent()` — the core agentic loop.

This function takes a Claude conversation and a set of MCP tools, lets Claude 
decide which tools to call, executes them via our MCP host, feeds results back, 
and loops until Claude has a final answer. Includes a `max_turns` safety limit 
to prevent runaway loops.

**Concepts demonstrated:** the agentic loop, tool scoping per agent, 
async tool execution.

In [5]:
agent_runner_code = '''"""
Agent runner for OurBrain.

The core agentic loop: send Claude a message with scoped MCP tools,
let Claude decide which to call, execute them via the MCP host, feed
results back, and repeat until Claude has a final answer.

This is the pattern that powers every agent system: Cursor, Claude Code,
LangChain agents, OpenAI Assistants. The implementation differs but the
loop is the same.
"""

import json
from anthropic import Anthropic
from agents.mcp_host import OurBrainMCPHost


def _mcp_tools_to_anthropic_format(mcp_tools, allowed_names=None):
    """
    Convert MCP tool definitions to Anthropic API format.
    
    If allowed_names is provided, filter to only those tools.
    This is the 'tool scoping per agent' pattern that makes
    multi-agent specialization work.
    """
    converted = []
    for t in mcp_tools:
        if allowed_names is not None and t.name not in allowed_names:
            continue
        converted.append({
            "name": t.name,
            "description": t.description,
            "input_schema": t.inputSchema,
        })
    return converted


async def run_agent(
    host: OurBrainMCPHost,
    anthropic_client: Anthropic,
    model: str,
    system_prompt: str,
    user_question: str,
    allowed_tools: list[str] | None = None,
    max_turns: int = 6,
    verbose: bool = True,
) -> str:
    """
    Run a single sub-agent: a Claude conversation with scoped MCP tools.
    
    Loops until Claude returns a final text answer, or until max_turns
    is hit (a safety net against runaway tool-calling loops).
    
    Returns Claude's final text response as a string.
    """
    all_mcp_tools = await host.list_tools()
    tools = _mcp_tools_to_anthropic_format(all_mcp_tools, allowed_names=allowed_tools)
    
    if verbose:
        scoped = [t["name"] for t in tools]
        print(f"  [agent has access to: {scoped}]")
    
    messages = [{"role": "user", "content": user_question}]
    
    for turn in range(max_turns):
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=2048,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )
        
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\\n".join(text_blocks)
        
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  [tool call: {block.name}({block.input})]")
                    result = await host.call_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            
            messages.append({"role": "user", "content": tool_results})
            continue
        
        return f"[unexpected stop_reason: {response.stop_reason}]"
    
    return "[hit max_turns without a final answer]"
'''

with open("../agents/agent_runner.py", "w") as f:
    f.write(agent_runner_code)

print("Wrote ../agents/agent_runner.py")

Wrote ../agents/agent_runner.py


In [6]:
# Force fresh imports in case anything is cached
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.agent_runner import run_agent
from anthropic import Anthropic
import json

# Load API key (in case this cell is run on its own)
with open("/Users/sagar/dev/TheBrain/secrets.json") as f:
    secrets = json.load(f)

anthropic_client = Anthropic(api_key=secrets["ANTHROPIC_API_KEY"])
MODEL = "claude-sonnet-4-5"  # if model-not-found, change to "claude-sonnet-4-20250514"

print("Setup complete ✓")

Setup complete ✓


In [7]:
answer = await run_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    system_prompt="You are a helpful assistant for the OurKitchen household app. Be concise.",
    user_question="What dietary restrictions does this household have?",
)

print("\n--- ANSWER ---")
print(answer)

  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'get_groceries']]


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: user messages must have non-empty content'}, 'request_id': 'req_011CaYkUKZrK4VYn5WfaV5iN'}

In [8]:
# Test each tool directly to see what they return
for tool_name in ["get_meals", "get_meal_history", "get_preferences", "get_groceries"]:
    result = await host.call_tool(tool_name)
    print(f"{tool_name}: type={type(result).__name__}, len={len(result) if result else 0}")
    print(f"  preview: {result[:100] if result else '(empty)'}")
    print()

get_meals: type=str, len=10346
  preview: [{"cuisine": "Vietnamese", "note": "", "fav": false, "rating": 5, "ts": 1774991541622, "proteins": [

get_meal_history: type=str, len=6452
  preview: [{"week_of": "2026-03-30", "meals": [{"date": "2026-03-31", "slot": "Dinner", "meal_name": "Unknown"

get_preferences: type=str, len=52
  preview: {"main": {"restrictions": ["no pork"], "loves": []}}

get_groceries: type=str, len=4121
  preview: [{"qty": "", "id": "mnq4qr1938f3un95ydq", "cat": "Other", "name": "Sunflower seeds", "checked": fals



In [9]:
agent_runner_code = '''"""
Agent runner for OurBrain.

The core agentic loop: send Claude a message with scoped MCP tools,
let Claude decide which to call, execute them via the MCP host, feed
results back, and repeat until Claude has a final answer.
"""

from anthropic import Anthropic
from agents.mcp_host import OurBrainMCPHost


def _mcp_tools_to_anthropic_format(mcp_tools, allowed_names=None):
    """Convert MCP tool definitions to Anthropic API format, optionally filtered."""
    converted = []
    for t in mcp_tools:
        if allowed_names is not None and t.name not in allowed_names:
            continue
        converted.append({
            "name": t.name,
            "description": t.description,
            "input_schema": t.inputSchema,
        })
    return converted


async def run_agent(
    host: OurBrainMCPHost,
    anthropic_client: Anthropic,
    model: str,
    system_prompt: str,
    user_question: str,
    allowed_tools: list[str] | None = None,
    max_turns: int = 6,
    verbose: bool = True,
) -> str:
    """
    Run a single sub-agent: a Claude conversation with scoped MCP tools.
    """
    all_mcp_tools = await host.list_tools()
    tools = _mcp_tools_to_anthropic_format(all_mcp_tools, allowed_names=allowed_tools)
    
    if verbose:
        scoped = [t["name"] for t in tools]
        print(f"  [agent has access to: {scoped}]")
    
    messages = [{"role": "user", "content": user_question}]
    
    for turn in range(max_turns):
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=2048,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )
        
        if verbose:
            print(f"  [turn {turn + 1}: stop_reason={response.stop_reason}]")
        
        # Case 1: Claude is done
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\\n".join(text_blocks)
        
        # Case 2: Claude wants to call tools
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  [tool call: {block.name}({block.input})]")
                    result = await host.call_tool(block.name, block.input)
                    
                    # Defensive: replace empty results with a sentinel so
                    # Claude knows the tool returned nothing rather than failing
                    if not result or not result.strip():
                        result = "(no data returned)"
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            
            # Defensive: if no tool calls were actually present despite the
            # stop_reason, bail rather than sending an empty user message
            if not tool_results:
                return "[stop_reason was tool_use but no tool calls were found]"
            
            messages.append({"role": "user", "content": tool_results})
            continue
        
        # Any other stop reason
        return f"[unexpected stop_reason: {response.stop_reason}]"
    
    return "[hit max_turns without a final answer]"
'''

with open("../agents/agent_runner.py", "w") as f:
    f.write(agent_runner_code)

print("Wrote updated ../agents/agent_runner.py")

Wrote updated ../agents/agent_runner.py


In [10]:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.agent_runner import run_agent

answer = await run_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    system_prompt="You are a helpful assistant for the OurKitchen household app. Be concise.",
    user_question="What dietary restrictions does this household have?",
)

print("\n--- ANSWER ---")
print(answer)

  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'get_groceries']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_preferences({})]
  [turn 2: stop_reason=end_turn]

--- ANSWER ---
This household has one dietary restriction: **no pork**.


In [11]:
# Test 2 — forces multiple tool calls
print("=" * 60)
print("TEST: multi-tool reasoning")
print("=" * 60)
answer = await run_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    system_prompt="You are a helpful assistant for the OurKitchen household app. Be concise.",
    user_question="What have we cooked the most over the last month, and does it align with our preferences?",
)
print(f"\nANSWER: {answer}\n")

TEST: multi-tool reasoning
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'get_groceries']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_meal_history({})]
  [tool call: get_preferences({})]
  [turn 2: stop_reason=end_turn]

ANSWER: Based on your meal history, here's what you've cooked the most over the last month:

**Most Frequent Meals:**
1. **Chicken Alfredo with Broccoli** - 2 times (Italian)
2. **Ground Beef Shawarma** - 2 times (Greek)
3. **Bibimbap** - 2 times (Asian)
4. **Chicken Quesadillas** - 2 times (Mexican)
5. **Pesto Chicken Pasta** - 2 times (Italian)

**Cuisine Breakdown:**
- Italian: 4 meals
- Asian/Greek/Mexican: 2 meals each

**Protein Breakdown:**
- Chicken: 6 meals
- Beef: 5 meals
- Seafood (Fish/Shrimp): 1 meal

**Alignment with Preferences:**
✅ **Yes, this aligns perfectly!** Your household restriction is "no pork," and none of the meals you've cooked contain pork. You've focused primarily on chicken and beef, which fits well w

In [ ]:
## Piece 3 — Specialized Sub-Agents

Three agents, each a thin wrapper around `run_agent` with a specialized system 
prompt and a scoped subset of MCP tools.

**Concepts demonstrated:** agent specialization through prompt engineering, 
tool scoping, the orchestrator-worker pattern (workers built here, orchestrator 
in piece 4).

In [12]:
sub_agents_code = '''"""
Specialized sub-agents for OurBrain.

Each sub-agent is a thin wrapper around run_agent with:
- A specialized system prompt that defines its expertise and reasoning style
- A scoped subset of MCP tools (only those relevant to its domain)

The thinness is the point. The agentic loop is generic. Specialization 
happens through prompts and tool scoping, not new code.
"""

from anthropic import Anthropic
from agents.mcp_host import OurBrainMCPHost
from agents.agent_runner import run_agent


# ── Nutrition Agent ────────────────────────────────────────────────
NUTRITION_SYSTEM_PROMPT = """You are the Nutrition Agent for OurBrain, a household intelligence system.

Your job is to analyze the household's eating patterns from a health and nutrition lens.
You reason about:
- Protein variety and distribution (red meat, poultry, fish, plant-based)
- Cuisine variety (are they eating monotonously or diversely?)
- Patterns over time (e.g., heavy meat weeks vs. lighter weeks)
- Adherence to dietary restrictions and preferences

When asked a question:
1. Use the available tools to fetch relevant data
2. Analyze the data quantitatively where possible (counts, percentages, trends)
3. Give a concise, evidence-based answer with specific examples from the data

Be direct. Don't pad with caveats. If patterns are concerning, say so. If they're 
healthy, say so. If you don't have enough data to answer, say that too.
"""

NUTRITION_TOOLS = ["get_meals", "get_meal_history", "get_preferences"]


async def run_nutrition_agent(
    host: OurBrainMCPHost,
    anthropic_client: Anthropic,
    model: str,
    user_question: str,
    verbose: bool = True,
) -> str:
    """Analyze nutritional patterns and protein/cuisine variety."""
    if verbose:
        print(f"\\n[NUTRITION AGENT] Question: {user_question}")
    return await run_agent(
        host=host,
        anthropic_client=anthropic_client,
        model=model,
        system_prompt=NUTRITION_SYSTEM_PROMPT,
        user_question=user_question,
        allowed_tools=NUTRITION_TOOLS,
        verbose=verbose,
    )


# ── Planner Agent ──────────────────────────────────────────────────
PLANNER_SYSTEM_PROMPT = """You are the Planner Agent for OurBrain, a household intelligence system.

Your job is to help plan upcoming meals based on:
- What the household has already cooked recently (avoid repetition)
- What's already in the grocery list (use what's on hand)
- The household's dietary restrictions and preferences

When asked to plan or suggest meals for a time period:
1. Check recent meal history to avoid suggesting recently-cooked meals
2. Check the grocery list to see what ingredients are already available
3. Respect all dietary restrictions strictly
4. Suggest meals from the existing meal library when possible (use get_meals)
5. Give specific meal names with brief reasoning ("Lemon Chicken — light protein, 
   you haven't had chicken in 8 days, lemons are on your grocery list")

Be practical. Plans should be actionable, not aspirational.
"""

PLANNER_TOOLS = ["get_meals", "get_meal_history", "get_groceries", "get_preferences"]


async def run_planner_agent(
    host: OurBrainMCPHost,
    anthropic_client: Anthropic,
    model: str,
    user_question: str,
    verbose: bool = True,
) -> str:
    """Plan upcoming meals based on history, groceries, and preferences."""
    if verbose:
        print(f"\\n[PLANNER AGENT] Question: {user_question}")
    return await run_agent(
        host=host,
        anthropic_client=anthropic_client,
        model=model,
        system_prompt=PLANNER_SYSTEM_PROMPT,
        user_question=user_question,
        allowed_tools=PLANNER_TOOLS,
        verbose=verbose,
    )


# ── Suggestion Agent ───────────────────────────────────────────────
SUGGESTION_SYSTEM_PROMPT = """You are the Suggestion Agent for OurBrain, a household intelligence system.

Your job is to recommend NEW meals or directions the household might enjoy, based on:
- Patterns in what they've cooked and rated highly
- Cuisines and proteins they seem to like
- Their dietary restrictions and preferences

You're the creative one. Your suggestions should:
- Build on what they already enjoy (if they love Vietnamese, suggest more SE Asian variety)
- Introduce gentle novelty (not radical departures from their taste)
- Be specific (don't say "try something new" — say "try Larb Gai, a Thai larb that 
  builds on your Vietnamese inclinations")
- Respect dietary restrictions absolutely

When making suggestions:
1. Pull data on existing meal library and history to identify patterns
2. Identify gaps (e.g., "you have lots of chicken dishes but few seafood")
3. Recommend specific dishes by name with brief reasoning

Be enthusiastic but grounded. You're a knowledgeable friend, not a hype machine.
"""

SUGGESTION_TOOLS = ["get_meals", "get_meal_history", "get_preferences"]


async def run_suggestion_agent(
    host: OurBrainMCPHost,
    anthropic_client: Anthropic,
    model: str,
    user_question: str,
    verbose: bool = True,
) -> str:
    """Recommend new meals based on patterns and preferences."""
    if verbose:
        print(f"\\n[SUGGESTION AGENT] Question: {user_question}")
    return await run_agent(
        host=host,
        anthropic_client=anthropic_client,
        model=model,
        system_prompt=SUGGESTION_SYSTEM_PROMPT,
        user_question=user_question,
        allowed_tools=SUGGESTION_TOOLS,
        verbose=verbose,
    )
'''

with open("../agents/sub_agents.py", "w") as f:
    f.write(sub_agents_code)

print("Wrote ../agents/sub_agents.py")

Wrote ../agents/sub_agents.py


In [13]:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.sub_agents import (
    run_nutrition_agent,
    run_planner_agent,
    run_suggestion_agent,
)

print("Sub-agents loaded ✓")

Sub-agents loaded ✓


In [14]:
#Test Nutrition Agent
answer = await run_nutrition_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    user_question="What are our protein and cuisine patterns over the last few weeks?",
)
print("\n--- NUTRITION AGENT ANSWER ---")
print(answer)


[NUTRITION AGENT] Question: What are our protein and cuisine patterns over the last few weeks?
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_meal_history({})]
  [turn 2: stop_reason=end_turn]

--- NUTRITION AGENT ANSWER ---
Based on your recent meal history, here's what the data shows:

## Protein Patterns

**Heavy chicken dominance**: 6 out of 11 meals (55%) featured chicken. You had:
- Chicken Alfredo with Broccoli (2x)
- Chicken quesadillas (2x)
- Pesto chicken pasta (2x)

**Red meat**: 4 meals (36%) with beef - all ground beef or beef bulgogi in:
- Ground Beef Shawarma (2x)
- Bibimbap (2x)

**Seafood**: Only 1 meal (9%) - the Sushi Bake with salmon and shrimp on April 3rd

**Plant-based proteins**: Zero dedicated plant-based protein meals

## Cuisine Patterns

**Italian heavy**: 4 meals (36%) - mostly cream-based pasta dishes

**Asian**: 2 meals (18%) - Korean bibimbap

**Mexican**: 2 meals (18%) - 

In [15]:
answer = await run_planner_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    user_question="What should we cook this week? Suggest 3 meals.",
)
print("\n--- PLANNER AGENT ANSWER ---")
print(answer)


[PLANNER AGENT] Question: What should we cook this week? Suggest 3 meals.
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'get_groceries']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_meal_history({})]
  [tool call: get_groceries({})]
  [tool call: get_preferences({})]
  [tool call: get_meals({})]
  [turn 2: stop_reason=end_turn]

--- PLANNER AGENT ANSWER ---
Based on your recent meal history, groceries, and preferences, here are **3 meal suggestions for this week**:

---

## 1. **Salmon and Roasted Veggies**
**Why:** You haven't had salmon or seafood in over a week (last was April 3rd with sushi bake). You already have **broccoli** and **shitake mushrooms** on hand (checked off your grocery list), and this is a simple, healthy protein change from all the chicken you've been eating.

---

## 2. **Chana Masala with Spinach**
**Why:** Great vegetarian option with bold flavors—you haven't cooked Indian food since March. You have **spinach** unchecked o

In [16]:
answer = await run_suggestion_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    user_question="What new meals should we try? Find gaps in our patterns.",
)
print("\n--- SUGGESTION AGENT ANSWER ---")
print(answer)


[SUGGESTION AGENT] Question: What new meals should we try? Find gaps in our patterns.
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_meals({})]
  [tool call: get_meal_history({})]
  [tool call: get_preferences({})]
  [turn 2: stop_reason=end_turn]

--- SUGGESTION AGENT ANSWER ---
Great! I've analyzed your meal patterns and I can see some clear trends. Let me break down what I'm seeing and suggest some exciting new directions:

## **What You Love** 🎯
- **Heavy Asian influence**: Vietnamese, Japanese, Korean, Indian (5-star ratings for Banh Mi, Sushi Bake, and Chicken Tikka Masala)
- **Chicken is your go-to protein**: It appears in 9 out of 16 meals
- **Comfort with bold flavors**: Chipotle, gochujang, tikka masala, pesto
- **No pork restriction** (which I'll respect absolutely)

## **The Gaps I Found** 🔍

### 1. **Underutilized Proteins**
- **Pork**: You have zero pork dishes! Given your love of Asian fla

In [17]:
sub_agents_code = '''"""
Specialized sub-agents for OurBrain.

Each sub-agent is a thin wrapper around run_agent with:
- A specialized system prompt that defines its expertise and reasoning style
- A scoped subset of MCP tools
- Hard guardrails for dietary restrictions to prevent recommendation violations
"""

from anthropic import Anthropic
from agents.mcp_host import OurBrainMCPHost
from agents.agent_runner import run_agent


# ── Shared guardrail block ─────────────────────────────────────────
# Injected into every agent's system prompt so dietary restrictions
# can never be misread as "gaps to fill" or "underutilized proteins".
HARD_RULES = """
══════════════════════════════════════════════════════════════════
HARD RULES — THESE OVERRIDE ANY OTHER REASONING
══════════════════════════════════════════════════════════════════

When you call get_preferences, the response will include a list of restrictions.
A restriction is an ABSOLUTE PROHIBITION, not a gap or an underutilized category.

For each item in the restrictions list, you MUST:
  1. NEVER recommend any dish containing that ingredient or its variants.
  2. NEVER frame it as a 'missed opportunity' or 'protein gap'.
  3. NEVER suggest 'building on existing tastes' with restricted ingredients.
  4. If a restriction says "no pork", that includes: pork, ham, bacon, prosciutto,
     pancetta, chorizo (pork-based), sausage (pork-based), pepperoni, salami
     (pork-based), pulled pork, ribs, lard, char siu, bulgogi (pork-based),
     and any other pork derivative.

If you find yourself writing about a restricted ingredient, STOP and remove it.
If a recipe in get_meals contains a restricted ingredient, do not recommend it
even if it has a high rating.

Violating these rules is a critical failure. Restrictions are non-negotiable.
══════════════════════════════════════════════════════════════════
"""


# ── Nutrition Agent ────────────────────────────────────────────────
NUTRITION_SYSTEM_PROMPT = f"""You are the Nutrition Agent for OurBrain, a household intelligence system.

Your job is to analyze the household's eating patterns from a health and nutrition lens.
You reason about:
- Protein variety and distribution (red meat, poultry, fish, plant-based)
- Cuisine variety (are they eating monotonously or diversely?)
- Patterns over time (e.g., heavy meat weeks vs. lighter weeks)
- Adherence to dietary restrictions and preferences

When asked a question:
1. Use the available tools to fetch relevant data
2. Analyze the data quantitatively where possible (counts, percentages, trends)
3. Give a concise, evidence-based answer with specific examples from the data

Be direct. Don't pad with caveats. If patterns are concerning, say so. If they're 
healthy, say so. If you don't have enough data to answer, say that too.

{HARD_RULES}
"""

NUTRITION_TOOLS = ["get_meals", "get_meal_history", "get_preferences"]


async def run_nutrition_agent(host, anthropic_client, model, user_question, verbose=True):
    """Analyze nutritional patterns and protein/cuisine variety."""
    if verbose:
        print(f"\\n[NUTRITION AGENT] Question: {user_question}")
    return await run_agent(
        host=host, anthropic_client=anthropic_client, model=model,
        system_prompt=NUTRITION_SYSTEM_PROMPT, user_question=user_question,
        allowed_tools=NUTRITION_TOOLS, verbose=verbose,
    )


# ── Planner Agent ──────────────────────────────────────────────────
PLANNER_SYSTEM_PROMPT = f"""You are the Planner Agent for OurBrain, a household intelligence system.

Your job is to help plan upcoming meals based on:
- What the household has already cooked recently (avoid repetition)
- What's already in the grocery list (use what's on hand)
- The household's dietary restrictions and preferences

When asked to plan or suggest meals for a time period:
1. ALWAYS call get_preferences first and read the restrictions list carefully
2. Check recent meal history to avoid suggesting recently-cooked meals
3. Check the grocery list to see what ingredients are already available
4. Suggest meals from the existing meal library when possible (use get_meals)
5. Give specific meal names with brief reasoning ("Lemon Chicken — light protein, 
   you haven't had chicken in 8 days, lemons are on your grocery list")

Be practical. Plans should be actionable, not aspirational.

{HARD_RULES}
"""

PLANNER_TOOLS = ["get_meals", "get_meal_history", "get_groceries", "get_preferences"]


async def run_planner_agent(host, anthropic_client, model, user_question, verbose=True):
    """Plan upcoming meals based on history, groceries, and preferences."""
    if verbose:
        print(f"\\n[PLANNER AGENT] Question: {user_question}")
    return await run_agent(
        host=host, anthropic_client=anthropic_client, model=model,
        system_prompt=PLANNER_SYSTEM_PROMPT, user_question=user_question,
        allowed_tools=PLANNER_TOOLS, verbose=verbose,
    )


# ── Suggestion Agent ───────────────────────────────────────────────
SUGGESTION_SYSTEM_PROMPT = f"""You are the Suggestion Agent for OurBrain, a household intelligence system.

Your job is to recommend NEW meals or directions the household might enjoy, based on:
- Patterns in what they've cooked and rated highly
- Cuisines and proteins they seem to like
- Their dietary restrictions and preferences

You're the creative one. Your suggestions should:
- Build on what they already enjoy
- Introduce gentle novelty (not radical departures)
- Be specific (give dish names, not categories)
- ABSOLUTELY respect dietary restrictions

CRITICAL workflow:
1. ALWAYS call get_preferences FIRST. Read the restrictions list and treat each
   restriction as an ABSOLUTE BAN, not a gap to fill.
2. Then call get_meals and get_meal_history to understand patterns
3. Identify gaps ONLY in non-restricted categories
4. Recommend specific dishes by name with brief reasoning

When in doubt about whether an ingredient is restricted, exclude it.

Be enthusiastic but grounded. You're a knowledgeable friend, not a hype machine.

{HARD_RULES}
"""

SUGGESTION_TOOLS = ["get_meals", "get_meal_history", "get_preferences"]


async def run_suggestion_agent(host, anthropic_client, model, user_question, verbose=True):
    """Recommend new meals based on patterns and preferences."""
    if verbose:
        print(f"\\n[SUGGESTION AGENT] Question: {user_question}")
    return await run_agent(
        host=host, anthropic_client=anthropic_client, model=model,
        system_prompt=SUGGESTION_SYSTEM_PROMPT, user_question=user_question,
        allowed_tools=SUGGESTION_TOOLS, verbose=verbose,
    )
'''

with open("../agents/sub_agents.py", "w") as f:
    f.write(sub_agents_code)

print("Wrote updated ../agents/sub_agents.py")

Wrote updated ../agents/sub_agents.py


In [18]:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.sub_agents import run_suggestion_agent

answer = await run_suggestion_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    user_question="What new meals should we try? Find gaps in our patterns.",
)
print("\n--- SUGGESTION AGENT ANSWER (with hard rules) ---")
print(answer)


[SUGGESTION AGENT] Question: What new meals should we try? Find gaps in our patterns.
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_preferences({})]
  [tool call: get_meals({})]
  [tool call: get_meal_history({})]
  [turn 2: stop_reason=end_turn]

--- SUGGESTION AGENT ANSWER (with hard rules) ---
Great! I've analyzed your preferences, recipe collection, and cooking history. Here's what I'm seeing—and some exciting recommendations to expand your repertoire:

## **Pattern Analysis**

**Strong Preferences:**
- **Chicken dominates** (8/16 recipes, frequently cooked) — you clearly love it and it's versatile
- **Italian & Mexican** cuisines appear often and work well for you
- **Bold, flavorful profiles**: Indian spices, Korean gochujang, Vietnamese herbs, chipotle
- **Seafood success**: You have salmon, shrimp, and flounder, and they're well-rounded
- **Balance seekers**: You mix indulgent (Alfredo) with lig

In [19]:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.sub_agents import (
    run_nutrition_agent,
    run_planner_agent,
    run_suggestion_agent,
)
print("Sub-agents reloaded ✓")

Sub-agents reloaded ✓


In [20]:
answer = await run_nutrition_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    user_question="What are our protein and cuisine patterns over the last few weeks?",
)
print("\n--- NUTRITION AGENT ANSWER ---")
print(answer)


[NUTRITION AGENT] Question: What are our protein and cuisine patterns over the last few weeks?
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_meal_history({})]
  [tool call: get_preferences({})]
  [turn 2: stop_reason=end_turn]

--- NUTRITION AGENT ANSWER ---
## Protein Patterns

**Heavy chicken dominance:** 6 out of 13 meals (46%) featured chicken. This is your go-to protein by far.

**Breakdown:**
- **Chicken:** 6 meals (46%)
- **Beef:** 4 meals (31%)
- **Seafood:** 1 meal with salmon + shrimp (8%)
- **Unknown/No protein logged:** 3 meals (23%)

**Key observations:**
- No plant-based proteins at all in the tracked period
- Seafood is severely underrepresented (only 1 meal in 2+ weeks)
- You're eating chicken 2x more often than beef
- Chicken appears in every cuisine type you cook (Italian, Mexican, Asian)

## Cuisine Patterns

**Reasonable variety across 5 cuisines:**
- **Italian:** 4 meals (31%) - Chi

In [21]:
# Force fresh imports to make sure we're using current code
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.mcp_host import OurBrainMCPHost
host = OurBrainMCPHost(server_path="../mcp_server/ourbrain_server.py")

# List tools — should now show 5
tools = await host.list_tools()
for t in tools:
    print(f"- {t.name}: {t.description[:80]}...")

- get_meals: Get all meals stored in OurKitchen, including name, cuisine, proteins, and ingre...
- get_meal_history: Get the full weekly meal history, with each planned meal enriched with its recip...
- get_preferences: Get household dietary preferences and restrictions....
- get_groceries: Get the current grocery list....
- search_meals_semantically: Search meals by semantic similarity for fuzzy/qualitative queries.
    
    Use ...


In [22]:
result = await host.call_tool(
    "search_meals_semantically",
    {"query": "something light and fresh", "n_results": 3}
)
print(result)

[{"name": "Bibimbap", "distance": 0.3126, "chunk": "Meal: Bibimbap. Cuisine: Asian. Ingredients: cooked white rice, beef bulgogi or marinated beef strips, spinach, bean sprouts, carrots, shiitake mushrooms, eggs, sesame oil, gochujang (Korean chili paste)"}, {"name": "Salmon and Roasted Veggies", "distance": 0.3136, "chunk": "Meal: Salmon and Roasted Veggies. Cuisine: American. Ingredients: salmon fillets, broccoli florets, olive oil, lemon, Asparagus, Baby potatoes"}, {"name": "Bahn Mi Subs", "distance": 0.3193, "chunk": "Meal: Bahn Mi Subs. Cuisine: Vietnamese. Ingredients: chicken breast, Vietnamese baguettes, mayonnaise, cucumber, pickled daikon radish, pickled carrots, fresh cilantro, jalape\u00f1o peppers, soy sauce"}]


In [23]:
# Read the current file
with open("../agents/sub_agents.py") as f:
    content = f.read()

# Update the tool lists
content = content.replace(
    'NUTRITION_TOOLS = ["get_meals", "get_meal_history", "get_preferences"]',
    'NUTRITION_TOOLS = ["get_meals", "get_meal_history", "get_preferences", "search_meals_semantically"]'
)
content = content.replace(
    'PLANNER_TOOLS = ["get_meals", "get_meal_history", "get_groceries", "get_preferences"]',
    'PLANNER_TOOLS = ["get_meals", "get_meal_history", "get_groceries", "get_preferences", "search_meals_semantically"]'
)
content = content.replace(
    'SUGGESTION_TOOLS = ["get_meals", "get_meal_history", "get_preferences"]',
    'SUGGESTION_TOOLS = ["get_meals", "get_meal_history", "get_preferences", "search_meals_semantically"]'
)

with open("../agents/sub_agents.py", "w") as f:
    f.write(content)

print("Updated tool scopes in sub_agents.py")

Updated tool scopes in sub_agents.py


In [24]:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.sub_agents import run_suggestion_agent

# A deliberately vibes-based question — no exact filter would answer this well
answer = await run_suggestion_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    user_question="We want something light and fresh for a hot day. What should we make?",
)
print("\n--- ANSWER ---")
print(answer)


[SUGGESTION AGENT] Question: We want something light and fresh for a hot day. What should we make?
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'search_meals_semantically']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_preferences({})]
  [tool call: search_meals_semantically({'query': 'light and fresh for a hot day', 'n_results': 5})]
  [turn 2: stop_reason=end_turn]

--- ANSWER ---
Perfect! Here are some great light and fresh options for a hot day:

## Top Recommendations:

**🥖 Banh Mi Subs** (Vietnamese)  
This is an excellent hot-day choice! The Vietnamese baguette sandwiches are bright, crunchy, and refreshing with pickled veggies, fresh cilantro, cucumber, and jalapeño. The chicken is typically grilled or quick-cooked, so minimal heat in the kitchen. The pickled daikon and carrots add that crisp, tangy element that's perfect when it's warm out.

**🥗 Ground Beef Shawarma** (Greek/Middle Eastern)  
While it involves cooked meat, this is served 

In [25]:
answer = await run_suggestion_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    user_question="What Vietnamese dishes should we add to our rotation?",
)
print("\n--- ANSWER ---")
print(answer)


[SUGGESTION AGENT] Question: What Vietnamese dishes should we add to our rotation?
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'search_meals_semantically']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_preferences({})]
  [tool call: get_meals({})]
  [tool call: get_meal_history({})]
  [turn 2: stop_reason=end_turn]

--- ANSWER ---
Great! Now I have a clear picture. Let me analyze what I see:

**Current patterns:**
- ✅ You have **one Vietnamese dish** (Banh Mi Subs - rated 5 stars!)
- 🚫 **No pork restriction** - this is CRITICAL for Vietnamese cuisine
- You love: Asian flavors (Bibimbap, Sushi Bake), chicken dishes, bold seasonings
- High ratings across the board for flavorful, well-seasoned meals

**Vietnamese dishes to add:**

1. **Pho Ga (Vietnamese Chicken Pho)** 🍜
   - You clearly love your Banh Mi (5-star rating!), and pho is another Vietnamese staple
   - Uses chicken, which is your most-cooked protein
   - Similar aromatic profile (star an